# Analyse de la régularité des TGV 🚆

## Objectif
Ce projet vise à analyser les retards des TGV en France à partir des données open data de la SNCF.

Nous cherchons à identifier les facteurs influençant la ponctualité des trains.

## Chargement des données

Nous utilisons un dataset issu de l’open data SNCF contenant des informations mensuelles sur les liaisons TGV.

In [2]:
import pandas as pd

df = pd.read_csv("regularite-mensuelle-tgv-aqst.csv", sep=";")
df.head()

,Date,Service,Gare de départ,Gare d'arrivée,Durée moyenne du trajet,Nombre de circulations prévues,Nombre de trains annulés,Commentaire annulations,Nombre de trains en retard au départ,Retard moyen des trains en retard au départ,...,Nombre trains en retard > 15min,Retard moyen trains en retard > 15 (si liaison concurrencée par vol),Nombre trains en retard > 30min,Nombre trains en retard > 60min,Prct retard pour causes externes,Prct retard pour cause infrastructure,Prct retard pour cause gestion trafic,Prct retard pour cause matériel roulant,Prct retard pour cause gestion en gare et réutilisation de matériel,"Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)"
0,2018-01,National,BORDEAUX ST JEAN,PARIS MONTPARNASSE,141,870,5,NaN,289,11.247809,...,110,6.511118,44,8,36.134454,31.092437,10.924370,15.966387,5.042017,0.840336
1,2018-01,National,LE MANS,PARIS MONTPARNASSE,56,406,1,NaN,213,8.479969,...,32,5.363539,9,4,20.000000,35.000000,16.666667,16.666667,8.333333,3.333333
2,2018-01,National,PARIS MONTPARNASSE,LA ROCHELLE VILLE,166,226,0,NaN,21,6.239683,...,11,2.938053,6,1,22.222222,27.777778,16.666667,16.666667,5.555556,11.111111
3,2018-01,National,PARIS MONTPARNASSE,NANTES,124,508,3,NaN,71,7.235211,...,39,5.292211,18,8,33.333333,22.222222,16.666667,20.370370,5.555556,1.851852
4,2018-01,National,POITIERS,PARIS MONTPARNASSE,94,472,4,NaN,224,6.784673,...,42,4.882372,10,0,15.789474,45.614035,19.298246,15.789474,1.754386,1.754386


## Exploration des données

Nous examinons la structure du dataset afin de comprendre les variables disponibles.

In [3]:
df.columns

Index(['Date', 'Service', 'Gare de départ', 'Gare d'arrivée',
       'Durée moyenne du trajet', 'Nombre de circulations prévues',
       'Nombre de trains annulés', 'Commentaire annulations',
       'Nombre de trains en retard au départ',
       'Retard moyen des trains en retard au départ',
       'Retard moyen de tous les trains au départ',
       'Commentaire retards au départ',
       'Nombre de trains en retard à l'arrivée',
       'Retard moyen des trains en retard à l'arrivée',
       'Retard moyen de tous les trains à l'arrivée',
       'Commentaire retards à l'arrivée', 'Nombre trains en retard > 15min',
       'Retard moyen trains en retard > 15 (si liaison concurrencée par vol)',
       'Nombre trains en retard > 30min', 'Nombre trains en retard > 60min',
       'Prct retard pour causes externes',
       'Prct retard pour cause infrastructure',
       'Prct retard pour cause gestion trafic',
       'Prct retard pour cause matériel roulant',
       'Prct retard pour cause ges

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11834 entries, 0 to 11833
Data columns (total 26 columns):
 #   Column                                                                                       Non-Null Count  Dtype  
---  ------                                                                                       --------------  -----  
 0   Date                                                                                         11834 non-null  object 
 1   Service                                                                                      11834 non-null  object 
 2   Gare de départ                                                                               11834 non-null  object 
 3   Gare d'arrivée                                                                               11834 non-null  object 
 4   Durée moyenne du trajet                                                                      11834 non-null  int64  
 5   Nombre de circulations prévues  

In [5]:
df.describe()

,Durée moyenne du trajet,Nombre de circulations prévues,Nombre de trains annulés,Commentaire annulations,Nombre de trains en retard au départ,Retard moyen des trains en retard au départ,Retard moyen de tous les trains au départ,Commentaire retards au départ,Nombre de trains en retard à l'arrivée,Retard moyen des trains en retard à l'arrivée,...,Nombre trains en retard > 15min,Retard moyen trains en retard > 15 (si liaison concurrencée par vol),Nombre trains en retard > 30min,Nombre trains en retard > 60min,Prct retard pour causes externes,Prct retard pour cause infrastructure,Prct retard pour cause gestion trafic,Prct retard pour cause matériel roulant,Prct retard pour cause gestion en gare et réutilisation de matériel,"Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)"
count,11834.000000,11834.000000,11834.000000,0.0,11834.000000,11834.000000,11834.000000,0.0,11834.000000,11834.000000,...,11834.000000,11834.000000,11834.000000,11834.000000,11834.000000,11834.000000,11834.000000,11834.000000,11834.000000,11834.000000
mean,170.401301,270.968396,8.653963,NaN,86.139175,12.264093,3.125096,NaN,37.361332,35.090215,...,26.787815,36.049226,12.498141,5.149907,21.555294,21.860600,20.341011,18.901051,7.374359,7.618523
std,87.668361,182.997263,22.636793,NaN,88.742772,11.799569,5.155105,NaN,31.133013,15.598450,...,22.588231,19.526490,12.056840,5.962474,15.970829,14.994220,14.647059,13.580690,8.108637,9.533479
min,0.000000,0.000000,0.000000,NaN,0.000000,0.000000,-229.269444,NaN,0.000000,-40.109259,...,0.000000,-4.000000,-44.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,99.000000,150.000000,0.000000,NaN,21.000000,6.143128,1.201985,NaN,15.000000,25.700000,...,11.000000,27.803487,4.000000,1.000000,10.526316,11.870265,10.344828,10.000000,0.000000,0.000000
50%,163.000000,229.000000,2.000000,NaN,52.000000,10.327290,2.314694,NaN,29.000000,33.406182,...,21.000000,37.456667,9.000000,3.000000,19.148936,20.000000,18.750000,17.073171,5.882353,5.000000
75%,222.750000,359.000000,7.000000,NaN,126.000000,15.700102,3.934298,NaN,51.000000,42.393099,...,36.000000,46.855774,17.000000,7.000000,30.000000,29.629630,28.571429,25.423729,11.111111,11.111111
max,786.000000,1100.000000,297.000000,NaN,596.000000,316.188095,115.047390,NaN,376.000000,299.600000,...,312.000000,299.600000,202.000000,71.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000


## Renommage des variables

Les noms de variables d'origine sont longs et peu lisibles.  
Afin de faciliter l'analyse et la manipulation des données, nous les simplifions.

Nous adoptons une convention :
- noms en minuscules
- utilisation de "_" au lieu des espaces
- noms courts mais explicites

In [6]:
df = df.rename(columns={
    "Date": "date",
    "Gare de départ": "gare_depart",
    "Gare d'arrivée": "gare_arrivee",
    "Nombre de circulations prévues": "nb_trains",
    "Nombre de trains annulés": "nb_annules",
    "Nombre de trains en retard à l'arrivée": "nb_retards",
    "Retard moyen de tous les trains à l'arrivée": "retard_moyen",
    "Nombre trains en retard > 15min": "retard_15min",
    "Nombre trains en retard > 30min": "retard_30min",
    "Prct retard pour causes externes": "cause_externe",
    "Prct retard pour cause infrastructure": "cause_infra",
    "Prct retard pour cause gestion trafic": "cause_trafic",
    "Prct retard pour cause matériel roulant": "cause_materiel"
})

df.head()

,date,Service,gare_depart,gare_arrivee,Durée moyenne du trajet,nb_trains,nb_annules,Commentaire annulations,Nombre de trains en retard au départ,Retard moyen des trains en retard au départ,...,retard_15min,Retard moyen trains en retard > 15 (si liaison concurrencée par vol),retard_30min,Nombre trains en retard > 60min,cause_externe,cause_infra,cause_trafic,cause_materiel,Prct retard pour cause gestion en gare et réutilisation de matériel,"Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)"
0,2018-01,National,BORDEAUX ST JEAN,PARIS MONTPARNASSE,141,870,5,NaN,289,11.247809,...,110,6.511118,44,8,36.134454,31.092437,10.924370,15.966387,5.042017,0.840336
1,2018-01,National,LE MANS,PARIS MONTPARNASSE,56,406,1,NaN,213,8.479969,...,32,5.363539,9,4,20.000000,35.000000,16.666667,16.666667,8.333333,3.333333
2,2018-01,National,PARIS MONTPARNASSE,LA ROCHELLE VILLE,166,226,0,NaN,21,6.239683,...,11,2.938053,6,1,22.222222,27.777778,16.666667,16.666667,5.555556,11.111111
3,2018-01,National,PARIS MONTPARNASSE,NANTES,124,508,3,NaN,71,7.235211,...,39,5.292211,18,8,33.333333,22.222222,16.666667,20.370370,5.555556,1.851852
4,2018-01,National,POITIERS,PARIS MONTPARNASSE,94,472,4,NaN,224,6.784673,...,42,4.882372,10,0,15.789474,45.614035,19.298246,15.789474,1.754386,1.754386


## Suppression des variables non pertinentes

Certaines variables ne sont pas utiles pour notre analyse :
- les colonnes de type "commentaire", qui sont textuelles et difficilement exploitables quantitativement
- certaines variables trop spécifiques ou redondantes

Nous les supprimons afin de simplifier le dataset et de nous concentrer sur les variables explicatives des retards.

df = df.drop(columns=[
    "Commentaire annulations",
    "Commentaire retards au départ",
    "Commentaire retards à l'arrivée"
], errors="ignore")

df.head()

## Transformation de la variable date

La variable "date" a été transformée en deux variables :
- "annee"
- "mois"

Cela permet de faciliter l’analyse temporelle.

La variable "date" initiale a ensuite été supprimée car elle n’apporte pas d’information supplémentaire.

In [10]:
# transformer la date
df["date"] = pd.to_datetime(df["date"])

# créer année et mois
df.insert(0, "annee", df["date"].dt.year)
df.insert(1, "mois", df["date"].dt.month)

# supprimer la colonne date
df = df.drop(columns=["date"])

df.head()

KeyError: 'date'